In [1]:

import requests
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser
from pathlib import Path

# Website configuration
BASE_URL = "https://www.allbirds.com/"
ROBOTS_URL = urljoin(BASE_URL, "robots.txt")
FEED_URL = urljoin(BASE_URL, "products.json")

USER_AGENT = (
    "DSC2107-Group9-Course-Project "
    "(contact: cynthiaageno@gmail.com)"
)

OUTPUT_DIR = Path("data/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def main():
    session = requests.Session()
    session.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "text/plain, application/json"
    })

    # STEP 1: Request robots.txt
    try:
        response = session.get(ROBOTS_URL, timeout=20)
        response.raise_for_status()

    except requests.exceptions.RequestException as error:
        print("Could not retrieve robots.txt.")
        print("Error:", error)
        print("STOP: Check your internet connection and try again.")
        return

    # STEP 2: Save robots.txt for the project report
    robots_file = OUTPUT_DIR / "robots_findings.txt"
    robots_file.write_text(response.text, encoding="utf-8")

    print("robots.txt saved to:", robots_file)

    # STEP 3: Check the website's crawling rules
    parser = RobotFileParser()
    parser.set_url(ROBOTS_URL)
    parser.parse(response.text.splitlines())

    allowed = parser.can_fetch(USER_AGENT, FEED_URL)

    print("Feed URL:", FEED_URL)
    print("Feed allowed for our User-Agent:", allowed)

    if not allowed:
        print("STOP: The robots rules disallow this feed.")
        return

    # STEP 4: Inspect a small sample of the product feed
    try:
        feed_response = session.get(
            FEED_URL,
            params={"limit": 5, "page": 1},
            timeout=20
        )

        print("Feed HTTP status:", feed_response.status_code)
        print(
            "Content type:",
            feed_response.headers.get("Content-Type")
        )

        feed_response.raise_for_status()

        # Confirm that the response contains JSON
        data = feed_response.json()

    except requests.exceptions.RequestException as error:
        print("Could not retrieve the product feed.")
        print("Error:", error)
        return

    except requests.exceptions.JSONDecodeError:
        print("The response was not valid JSON.")
        print("Inspect the response before continuing.")
        return

    # STEP 5: Inspect the structure of the response
    if not isinstance(data, dict):
        print("Unexpected response format:", type(data).__name__)
        return

    print("Top-level keys:", list(data.keys()))

    products = data.get("products", [])

    if not isinstance(products, list):
        print("Unexpected format for the products field.")
        return

    print("Sample product count:", len(products))

    if products:
        print("Sample product keys:", list(products[0].keys()))

        variants = products[0].get("variants") or []

        if variants:
            print("Sample variant keys:", list(variants[0].keys()))
        else:
            print("No variants found for the sample product.")
    else:
        print("No products returned. Check the feed response and parameters.")


if __name__ == "__main__":
    main()


robots.txt saved to: data\raw\robots_findings.txt
Feed URL: https://www.allbirds.com/products.json
Feed allowed for our User-Agent: True
Feed HTTP status: 200
Content type: application/json; charset=utf-8
Top-level keys: ['products']
Sample product count: 5
Sample product keys: ['id', 'title', 'handle', 'body_html', 'published_at', 'created_at', 'updated_at', 'vendor', 'product_type', 'tags', 'variants', 'images', 'options']
Sample variant keys: ['id', 'title', 'option1', 'option2', 'option3', 'sku', 'requires_shipping', 'taxable', 'featured_image', 'available', 'price', 'grams', 'compare_at_price', 'position', 'product_id', 'created_at', 'updated_at']
